# CP3 — Classificação de sentimentos com Transformers

**Disciplina:** ARTIFICIAL INTELLIGENCE e DEEP LEARNING APPLICADA
**Professor:** Dr. Alexandre Miguel de Carvalho
**Ferramentas:** Python 3.12 · 🤗 transformers · 🤗 datasets · PyTorch · scikit-learn

---

Neste checkpoint você vai fazer *fine-tuning* de um modelo de linguagem pré-treinado
para classificar o sentimento de textos. O notebook é um pipeline completo e genérico:
**você escolhe o modelo linguístico e o dataset** na célula de configuração, e todo o
resto se adapta.

```mermaid
flowchart LR
    A[1. Configuração<br/>modelo + dataset] --> B[2. Carga e<br/>padronização]
    B --> C[3. Análise<br/>exploratória]
    C --> D[4. Baseline<br/>TF-IDF + LogReg]
    D --> E[5. Tokenização]
    E --> F[6. Fine-tuning]
    F --> G[7. Avaliação<br/>no teste]
    G --> H[8. Análise<br/>de erros]
    H --> I[9. Inferência]
```

### O que você decide

| Decisão | Onde | Exemplos |
|---|---|---|
| Modelo linguístico | `MODELO` | BERTimbau, XLM-RoBERTa, DistilBERT… |
| Dataset | `FONTE_DADOS`, `DATASET_HUB` ou `CAMINHO_CSV` | um dataset do Hugging Face Hub ou um CSV seu |
| Colunas de texto e rótulo | `COLUNA_TEXTO`, `COLUNA_ROTULO` | `"text"`, `"review"`, `"label"`, `"stars"`… |
| Como os rótulos viram sentimento | `mapear_rotulo` | estrelas 1–5 → negativo / neutro / positivo |
| Orçamento de treino | `AMOSTRA_TREINO`, `EPOCAS`, `MAX_LEN` | menor em CPU, maior em GPU |

> **Regra de ouro:** modelo e dataset precisam falar a mesma língua. Um BERT treinado só
> em inglês vai mal em resenhas em português — e esse é um ótimo experimento para mostrar
> no relatório.

## 0. Ambiente

Se estiver no Google Colab, ative a GPU em *Ambiente de execução → Alterar o tipo de
ambiente de execução → GPU*. Em CPU o notebook também roda, desde que você mantenha
uma amostra de treino pequena.

In [ ]:
# Descomente se as bibliotecas não estiverem instaladas (ex.: Colab já tem quase tudo)
# %pip install -q "transformers>=4.46" "datasets>=3.0" accelerate scikit-learn seaborn

In [ ]:
import os
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from datasets import load_dataset, DatasetDict, ClassLabel
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer,
                          EarlyStoppingCallback, pipeline, set_seed)
import transformers

SEMENTE = 42
random.seed(SEMENTE)
np.random.seed(SEMENTE)
set_seed(SEMENTE)

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print(f"transformers {transformers.__version__} | torch {torch.__version__} | dispositivo: {DISPOSITIVO}")

## 1. Configuração — **as suas escolhas**

Esta é a única célula que você *precisa* alterar. Abaixo há um catálogo de modelos
sugeridos; você pode usar qualquer outro do [Hugging Face Hub](https://huggingface.co/models?pipeline_tag=text-classification)
que seja um *encoder* (BERT, RoBERTa, DeBERTa, ELECTRA, ALBERT…).

In [ ]:
# Catálogo de sugestões: apelido -> (id no Hub, idioma, observação)
CATALOGO_MODELOS = {
    "bertimbau-base":   ("neuralmind/bert-base-portuguese-cased", "pt", "BERT treinado em português brasileiro (110M)"),
    "bertimbau-large":  ("neuralmind/bert-large-portuguese-cased", "pt", "versão large (335M) — só com GPU"),
    "albertina-ptbr":   ("PORTULAN/albertina-100m-portuguese-ptbr-encoder", "pt", "DeBERTa em português brasileiro (100M)"),
    "xlm-roberta-base": ("FacebookAI/xlm-roberta-base", "multi", "multilíngue, 100 idiomas (280M)"),
    "mbert-distil":     ("distilbert/distilbert-base-multilingual-cased", "multi", "multilíngue e leve (135M)"),
    "distilbert-en":    ("distilbert/distilbert-base-uncased", "en", "inglês, leve e rápido (66M)"),
    "roberta-en":       ("FacebookAI/roberta-base", "en", "inglês (125M)"),
    "deberta-v3-en":    ("microsoft/deberta-v3-base", "en", "inglês, forte em classificação (184M)"),
}
pd.DataFrame(CATALOGO_MODELOS, index=["id no Hub", "idioma", "observação"]).T

In [ ]:
# ---------------------------------------------------------------------------
# 1) MODELO: um apelido do catálogo OU qualquer id do Hugging Face Hub
# ---------------------------------------------------------------------------
MODELO = "distilbert-en"
MODELO_ID = CATALOGO_MODELOS[MODELO][0] if MODELO in CATALOGO_MODELOS else MODELO

# ---------------------------------------------------------------------------
# 2) DATASET: "hub" (Hugging Face) ou "csv" (arquivo local / URL)
# ---------------------------------------------------------------------------
FONTE_DADOS = "hub"

# Se FONTE_DADOS == "hub"
DATASET_HUB = "stanfordnlp/imdb"   # resenhas de filmes em inglês (pos/neg)
DATASET_CONFIG = None              # subconjunto (config), ex.: "sentiment" no tweet_eval
DATASET_DIR = None                 # subpasta de arquivos, ex.: "portuguese"
DATASET_REVISION = None            # ex.: "refs/convert/parquet" (ver tabela de sugestões)

# Se FONTE_DADOS == "csv"
CAMINHO_CSV = "meu_dataset.csv"
SEPARADOR_CSV = ","

# Nome das colunas no dataset escolhido
COLUNA_TEXTO = "text"
COLUNA_ROTULO = "label"

# Nomes legíveis das classes, na ordem dos ids (None = usar os nomes do próprio dataset)
NOMES_CLASSES = ["negativo", "positivo"]

# ---------------------------------------------------------------------------
# 3) ORÇAMENTO DE TREINO
# ---------------------------------------------------------------------------
# Referência: com GPU (Colab T4), 4.000 textos x 2 épocas levam poucos minutos.
# Em CPU, cada texto de 256 tokens custa ~0,5 s por época: 2.000 x 2 épocas ≈ 30-40 min.
AMOSTRA_TREINO = 2000     # None = usar tudo
AMOSTRA_AVALIACAO = 1000  # tamanho máximo da validação e do teste
FRACAO_VALIDACAO = 0.1    # usada quando o dataset não traz validação
FRACAO_TESTE = 0.2        # usada quando o dataset não traz teste
MAX_LEN = None            # None = percentil 95 dos comprimentos (máx. 512 com GPU, 256 em CPU)
EPOCAS = 2
TAXA_APRENDIZADO = 2e-5
TAMANHO_LOTE = 16
PASTA_SAIDA = "modelo_sentimentos"

In [ ]:
def mapear_rotulo(valor):
    # Converte o rótulo original no rótulo de sentimento usado no treino.
    #
    # Deixe como está se o dataset já vem com o sentimento. Exemplos de adaptação:
    #   estrelas 1-5 -> 3 classes:  return 0 if valor <= 2 else (1 if valor == 3 else 2)
    #   estrelas 1-5 -> 2 classes:  return None if valor == 3 else int(valor >= 4)
    #   texto:                      return {"neg": 0, "pos": 1}.get(valor)
    # Retornar None descarta o exemplo.
    return valor

### Sugestões de datasets

| `DATASET_HUB` | Outros parâmetros | Idioma | `COLUNA_TEXTO`, `COLUNA_ROTULO` | Observação |
|---|---|---|---|---|
| `stanfordnlp/imdb` | — | en | `text`, `label` | 50 mil resenhas de filmes, 2 classes |
| `celsowm/imdb-reviews-pt-br` | — | pt | `texto`, `sentimento` | IMDB traduzido, 49 mil, 2 classes (0 = neg) |
| `AiresPucrs/google-play-apps-review-pt` | — | pt | `review`, `sentiment` | 20 mil avaliações de apps, 2 classes |
| `ruanchaves/b2w-reviews01` | `DATASET_REVISION="refs/convert/parquet"` | pt | `review_text`, `overall_rating` | 132 mil resenhas da Americanas com 1–5 estrelas: use `mapear_rotulo` |
| `cardiffnlp/tweet_sentiment_multilingual` | `DATASET_REVISION="refs/convert/parquet"`, `DATASET_DIR="portuguese"` | pt | `text`, `label` | 3 mil tweets, 3 classes (neg/neu/pos) |
| `cardiffnlp/tweet_eval` | `DATASET_CONFIG="sentiment"` | en | `text`, `label` | tweets, 3 classes |
| `fancyzhx/amazon_polarity` | — | en | `content`, `label` | 4 milhões de resenhas: use amostra |

> Se aparecer o erro *"Dataset scripts are no longer supported"*, o dataset usa um script
> de carga antigo. Quase sempre funciona apontando para a versão em Parquet que o Hub gera
> automaticamente: `DATASET_REVISION = "refs/convert/parquet"` (e `DATASET_DIR` com o
> subconjunto, quando houver).

Também vale um CSV próprio (coletado por você, Kaggle etc.): basta ter uma coluna de
texto e uma de rótulo. Confira sempre a página do dataset no Hub para saber os nomes
das colunas e o significado de cada rótulo. Se você trocar o dataset, ajuste também
`NOMES_CLASSES` (ou use `None`) e as frases de teste da seção 9.

## 2. Carga e padronização do dataset

Cada dataset vem com um formato diferente. Aqui padronizamos tudo para três splits
(`train`, `validation`, `test`), cada um com as colunas `text` e `label` (inteiro).

In [ ]:
if FONTE_DADOS == "hub":
    bruto = load_dataset(DATASET_HUB, DATASET_CONFIG, data_dir=DATASET_DIR, revision=DATASET_REVISION)
elif FONTE_DADOS == "csv":
    bruto = load_dataset("csv", data_files=CAMINHO_CSV, sep=SEPARADOR_CSV)
else:
    raise ValueError("FONTE_DADOS deve ser 'hub' ou 'csv'")

bruto

In [ ]:
# Nomes de classe que o próprio dataset declara (se a coluna for ClassLabel)
_feature_rotulo = next(iter(bruto.values())).features[COLUNA_ROTULO]
NOMES_ORIGINAIS = _feature_rotulo.names if isinstance(_feature_rotulo, ClassLabel) else None

def padronizar(ds):
    ds = ds.select_columns([COLUNA_TEXTO, COLUNA_ROTULO])
    ds = ds.map(lambda ex: {"text": ex[COLUNA_TEXTO], "label_novo": mapear_rotulo(ex[COLUNA_ROTULO])},
                remove_columns=[COLUNA_TEXTO, COLUNA_ROTULO])
    ds = ds.filter(lambda ex: ex["label_novo"] is not None and ex["text"] is not None
                   and str(ex["text"]).strip() != "")
    return ds.rename_column("label_novo", "label")

ds = DatasetDict({nome: padronizar(split) for nome, split in bruto.items()})

# Alguns datasets usam -1 no teste para "sem rótulo" (ex.: GLUE). Esses exemplos não servem.
ds = ds.filter(lambda ex: not (isinstance(ex["label"], int) and ex["label"] < 0))
ds = DatasetDict({nome: split for nome, split in ds.items() if len(split) > 0})   # ex.: "unsupervised" do IMDB

# Rótulos em texto ("pos", "neg") viram ids; rótulos inteiros ganham um ClassLabel
todos_rotulos = sorted({r for split in ds.values() for r in split.unique("label")}, key=str)
if all(isinstance(r, (int, np.integer)) for r in todos_rotulos):
    n = int(max(todos_rotulos)) + 1
    nomes = NOMES_CLASSES or (NOMES_ORIGINAIS if NOMES_ORIGINAIS and len(NOMES_ORIGINAIS) == n else None) \
            or [str(i) for i in range(n)]
else:
    nomes = NOMES_CLASSES or [str(r) for r in todos_rotulos]
    para_id = {r: i for i, r in enumerate(todos_rotulos)}
    ds = ds.map(lambda ex: {"label": para_id[ex["label"]]})
ds = ds.cast_column("label", ClassLabel(names=list(nomes)))

ID2LABEL = dict(enumerate(nomes))
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_CLASSES = len(nomes)
print("Classes:", ID2LABEL)

In [ ]:
# Garante train / validation / test, com divisão estratificada quando faltar algum
if "train" not in ds:
    ds = DatasetDict({"train": next(iter(ds.values()))})
if "test" not in ds:
    partes = ds["train"].train_test_split(test_size=FRACAO_TESTE, stratify_by_column="label", seed=SEMENTE)
    ds["train"], ds["test"] = partes["train"], partes["test"]
if "validation" not in ds:
    partes = ds["train"].train_test_split(test_size=FRACAO_VALIDACAO, stratify_by_column="label", seed=SEMENTE)
    ds["train"], ds["validation"] = partes["train"], partes["test"]
ds = DatasetDict({k: ds[k] for k in ["train", "validation", "test"]})

def amostrar(split, n):
    # Amostra estratificada de até n exemplos
    if n is None or len(split) <= n:
        return split.shuffle(seed=SEMENTE)
    return split.train_test_split(train_size=n, stratify_by_column="label", seed=SEMENTE)["train"]

ds["train"] = amostrar(ds["train"], AMOSTRA_TREINO)
ds["validation"] = amostrar(ds["validation"], AMOSTRA_AVALIACAO)
ds["test"] = amostrar(ds["test"], AMOSTRA_AVALIACAO)
ds

## 3. Análise exploratória

Antes de treinar, responda: as classes estão balanceadas? Qual o tamanho típico dos
textos? Isso define a métrica principal (acurácia engana em dados desbalanceados) e o
`MAX_LEN` (textos maiores que ele são cortados).

In [ ]:
df_treino = ds["train"].to_pandas()
df_treino["classe"] = df_treino["label"].map(ID2LABEL)
df_treino["palavras"] = df_treino["text"].str.split().str.len()

for i, linha in df_treino.groupby("classe").head(2).iterrows():
    print(f"[{linha['classe']}] {linha['text'][:300]}\n")

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 4))
contagem = pd.concat({s: ds[s].to_pandas()["label"].map(ID2LABEL).value_counts() for s in ds}, axis=1)
contagem.plot.bar(ax=eixos[0], rot=0)
eixos[0].set_title("Exemplos por classe e split")
eixos[0].set_ylabel("exemplos")

sns.histplot(data=df_treino, x="palavras", hue="classe", bins=50, ax=eixos[1], element="step")
eixos[1].set_title("Tamanho dos textos (palavras) no treino")
plt.tight_layout()
plt.show()

print("Proporção das classes no treino:")
print(df_treino["classe"].value_counts(normalize=True).round(3).to_string())

## 4. Baseline: TF-IDF + Regressão Logística

Um transformer custa minutos ou horas de treino e centenas de MB de memória. Para saber
se ele vale a pena, precisamos de uma referência barata: um modelo de saco de palavras
treinado nos **mesmos** dados. Se o transformer não superar com folga o baseline, algo
está errado (ou o problema é simples).

In [ ]:
def metricas(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred),
            "f1_macro": f1_score(y_true, y_pred, average="macro"),
            "f1_weighted": f1_score(y_true, y_pred, average="weighted")}

inicio = time.time()
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50_000, sublinear_tf=True)
X_treino = tfidf.fit_transform(list(ds["train"]["text"]))
X_teste = tfidf.transform(list(ds["test"]["text"]))
y_treino = np.array(list(ds["train"]["label"]))
y_teste = np.array(list(ds["test"]["label"]))
baseline = LogisticRegression(max_iter=2000, class_weight="balanced")
baseline.fit(X_treino, y_treino)
pred_baseline = baseline.predict(X_teste)

resultado_baseline = metricas(y_teste, pred_baseline)
print(f"Baseline treinado em {time.time() - inicio:.1f}s")
print({k: round(v, 4) for k, v in resultado_baseline.items()})

## 5. Tokenização

O tokenizador **pertence ao modelo**: cada modelo foi pré-treinado com um vocabulário
próprio (WordPiece, BPE, SentencePiece). Por isso carregamos os dois pelo mesmo
`MODELO_ID`. Vamos ver como o modelo escolhido quebra uma frase.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)

frase = ds["train"][0]["text"][:120]
print("Texto: ", frase)
print("Tokens:", tokenizer.tokenize(frase))
print(f"Vocabulário: {tokenizer.vocab_size:,} tokens | limite do modelo: {tokenizer.model_max_length}")

In [ ]:
# Comprimento em tokens (não em palavras!) para escolher o MAX_LEN
amostra_len = ds["train"].select(range(min(2000, len(ds["train"]))))
n_tokens = np.array([len(ids) for ids in tokenizer(list(amostra_len["text"]), truncation=False)["input_ids"]])

limite_modelo = min(tokenizer.model_max_length, 512 if DISPOSITIVO == "cuda" else 256)
if MAX_LEN is None:
    MAX_LEN = int(min(limite_modelo, np.percentile(n_tokens, 95)))
    MAX_LEN = max(32, int(np.ceil(MAX_LEN / 8) * 8))   # múltiplo de 8 é mais eficiente na GPU

print(f"Tokens por texto: mediana {np.median(n_tokens):.0f} | p95 {np.percentile(n_tokens, 95):.0f} | máx {n_tokens.max()}")
print(f"MAX_LEN = {MAX_LEN} -> {np.mean(n_tokens > MAX_LEN):.1%} dos textos serão truncados")

In [ ]:
def tokenizar(lote):
    return tokenizer(lote["text"], truncation=True, max_length=MAX_LEN)

ds_tok = ds.map(tokenizar, batched=True)
coletor = DataCollatorWithPadding(tokenizer=tokenizer)   # padding dinâmico por lote
ds_tok

## 6. Fine-tuning

`AutoModelForSequenceClassification` carrega o corpo pré-treinado e adiciona uma
**cabeça de classificação** nova (uma camada linear sobre o vetor do token `[CLS]`),
inicializada aleatoriamente — por isso o aviso de pesos "newly initialized" é esperado.
O *fine-tuning* ajusta a cabeça **e** o corpo com uma taxa de aprendizado pequena.

In [ ]:
modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_ID, num_labels=NUM_CLASSES, id2label=ID2LABEL, label2id=LABEL2ID,
)
n_param = sum(p.numel() for p in modelo.parameters())
print(f"{MODELO_ID}: {n_param / 1e6:.1f} M parâmetros")

In [ ]:
def calcular_metricas(avaliacao):
    logits, rotulos = avaliacao
    return metricas(rotulos, np.argmax(logits, axis=-1))

# Aquecimento: a taxa de aprendizado sobe de 0 ao valor alvo nos primeiros 10% dos passos
passos_totais = int(np.ceil(len(ds_tok["train"]) / TAMANHO_LOTE)) * EPOCAS

args = TrainingArguments(
    output_dir=PASTA_SAIDA,
    num_train_epochs=EPOCAS,
    learning_rate=TAXA_APRENDIZADO,
    per_device_train_batch_size=TAMANHO_LOTE,
    per_device_eval_batch_size=TAMANHO_LOTE * 2,
    weight_decay=0.01,
    warmup_steps=int(0.1 * passos_totais),
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=max(1, passos_totais // 20),
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEMENTE,
)

trainer = Trainer(
    model=modelo,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=coletor,
    compute_metrics=calcular_metricas,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

inicio = time.time()
trainer.train()
print(f"Treino concluído em {(time.time() - inicio) / 60:.1f} min")

In [ ]:
# Curvas de aprendizado a partir do histórico do Trainer
historico = pd.DataFrame(trainer.state.log_history)
perda_treino = historico.dropna(subset=["loss"]) if "loss" in historico else historico.iloc[:0]
aval = historico.dropna(subset=["eval_loss"])

fig, eixos = plt.subplots(1, 2, figsize=(13, 4))
eixos[0].plot(perda_treino["step"], perda_treino["loss"], label="treino")
eixos[0].plot(aval["step"], aval["eval_loss"], "o-", label="validação")
eixos[0].set(title="Perda", xlabel="passo")
eixos[0].legend()
eixos[1].plot(aval["epoch"], aval["eval_accuracy"], "o-", label="acurácia")
eixos[1].plot(aval["epoch"], aval["eval_f1_macro"], "o-", label="F1 macro")
eixos[1].set(title="Validação", xlabel="época")
eixos[1].legend()
plt.tight_layout()
plt.show()

## 7. Avaliação no conjunto de teste

O teste só é usado **uma vez**, depois de todas as escolhas feitas na validação.

In [ ]:
saida_teste = trainer.predict(ds_tok["test"])
probs_teste = torch.softmax(torch.tensor(saida_teste.predictions), dim=-1).numpy()
pred_teste = probs_teste.argmax(axis=1)

resultado_transformer = metricas(y_teste, pred_teste)
print(classification_report(y_teste, pred_teste, target_names=list(ID2LABEL.values()), digits=4))

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4.5))
for eixo, pred, titulo in [(eixos[0], pred_baseline, "Baseline TF-IDF"),
                           (eixos[1], pred_teste, f"Transformer ({MODELO})")]:
    cm = confusion_matrix(y_teste, pred, normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=eixo,
                xticklabels=ID2LABEL.values(), yticklabels=ID2LABEL.values())
    eixo.set(title=titulo, xlabel="previsto", ylabel="real")
plt.tight_layout()
plt.show()

comparacao = pd.DataFrame({"Baseline TF-IDF + LogReg": resultado_baseline,
                           f"Transformer ({MODELO_ID})": resultado_transformer}).T.round(4)
comparacao

## 8. Análise de erros

Métricas dizem *quanto* o modelo erra; os exemplos dizem *por quê*. Os erros mais
interessantes são aqueles em que o modelo estava **muito confiante** e errou: ironia,
negação ("não é ruim"), rótulo errado no próprio dataset, texto truncado…

In [ ]:
df_teste = ds["test"].to_pandas()
df_teste["real"] = df_teste["label"].map(ID2LABEL)
df_teste["previsto"] = pd.Series(pred_teste).map(ID2LABEL)
df_teste["confianca"] = probs_teste.max(axis=1)

erros = df_teste[df_teste["real"] != df_teste["previsto"]].sort_values("confianca", ascending=False)
print(f"{len(erros)} erros em {len(df_teste)} exemplos de teste\n")
for _, linha in erros.head(8).iterrows():
    print(f"real={linha['real']:<10} previsto={linha['previsto']:<10} confiança={linha['confianca']:.2f}")
    print(f"   {linha['text'][:300]}\n")

## 9. Inferência em textos novos

Salvamos o melhor modelo e o tokenizador juntos e os recarregamos com um `pipeline`,
exatamente como seriam usados em produção.

In [ ]:
trainer.save_model(PASTA_SAIDA)
tokenizer.save_pretrained(PASTA_SAIDA)

classificador = pipeline("text-classification", model=PASTA_SAIDA, tokenizer=PASTA_SAIDA,
                         device=0 if torch.cuda.is_available() else -1)

# Escreva frases no idioma do seu dataset — inclua casos difíceis (ironia, negação, misto)
frases = [
    "This movie was absolutely wonderful, I loved every minute of it.",
    "What a waste of time. The plot made no sense at all.",
    "It was not bad, but I would not watch it again.",
    "Oh great, another two hours of my life I will never get back.",
]
for frase, resultado in zip(frases, classificador(frases)):
    print(f"{resultado['label']:<10} ({resultado['score']:.2f})  {frase}")

## 10. Sua análise

Responda nas células abaixo, com base nos resultados **do seu** modelo e **do seu** dataset:

1. **Escolhas.** Qual modelo e qual dataset você escolheu, e por quê? O idioma do modelo
   combina com o do dataset?
2. **Dados.** As classes estão balanceadas? Que métrica você considera a principal e por quê?
   Quantos textos foram truncados pelo `MAX_LEN`?
3. **Baseline × transformer.** Quanto o transformer ganhou sobre o TF-IDF? Esse ganho
   justifica o custo de treino e de inferência?
4. **Erros.** Que padrões aparecem nos erros mais confiantes? Algum parece rótulo errado
   do próprio dataset?
5. **Experimento.** Troque **uma** coisa (outro modelo, mais dados, mais épocas, `MAX_LEN`
   maior) e compare com o primeiro resultado. O que mudou?

*Escreva aqui a sua análise.*